# Neural Networks (MLP) - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (confusion_matrix, classification_report, 
                             accuracy_score, f1_score)
from sklearn.neural_network import MLPClassifier as SklearnMLP
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is a Neural Network?

A Neural Network (specifically a Multi-Layer Perceptron or MLP) is a computational model inspired by biological neural networks. It consists of layers of interconnected nodes (neurons) that can learn complex non-linear patterns in data.

---

### 1.1 The Perceptron

The **perceptron** is the simplest neural network unit - a single artificial neuron.

**Mathematical Definition:**
$$z = \sum_{i=1}^{n} w_i x_i + b = \mathbf{w}^T \mathbf{x} + b$$
$$\hat{y} = \phi(z)$$

Where:
- $\mathbf{x}$ = input features
- $\mathbf{w}$ = weights
- $b$ = bias term
- $\phi$ = activation function

---

### 1.2 Multi-Layer Architecture

A Multi-Layer Perceptron (MLP) consists of:

1. **Input Layer**: Receives the input features (no computation)
2. **Hidden Layers**: One or more layers that learn representations
3. **Output Layer**: Produces the final prediction

**Notation:**
- $L$ = total number of layers (including output, excluding input)
- $n^{[l]}$ = number of neurons in layer $l$
- $W^{[l]}$ = weight matrix for layer $l$, shape $(n^{[l]}, n^{[l-1]})$
- $b^{[l]}$ = bias vector for layer $l$, shape $(n^{[l]}, 1)$
- $a^{[l]}$ = activations of layer $l$

---

### 1.3 Forward Propagation

Forward propagation computes the output of the network layer by layer.

**For each layer $l = 1, 2, ..., L$:**

$$Z^{[l]} = W^{[l]} \cdot A^{[l-1]} + b^{[l]}$$
$$A^{[l]} = \phi^{[l]}(Z^{[l]})$$

Where $A^{[0]} = X$ (input data)

**Matrix dimensions (for batch of $m$ samples):**
- $X$: $(n_{features}, m)$
- $W^{[l]}$: $(n^{[l]}, n^{[l-1]})$
- $Z^{[l]}$: $(n^{[l]}, m)$
- $A^{[l]}$: $(n^{[l]}, m)$

---

### 1.4 Activation Functions

Activation functions introduce non-linearity, allowing networks to learn complex patterns.

#### Sigmoid
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
$$\sigma'(z) = \sigma(z)(1 - \sigma(z))$$

- **Range**: (0, 1)
- **Pros**: Smooth gradient, output interpretable as probability
- **Cons**: Vanishing gradient for large |z|, outputs not zero-centered

#### Tanh (Hyperbolic Tangent)
$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$
$$\tanh'(z) = 1 - \tanh^2(z)$$

- **Range**: (-1, 1)
- **Pros**: Zero-centered outputs, stronger gradients than sigmoid
- **Cons**: Still suffers from vanishing gradient

#### ReLU (Rectified Linear Unit)
$$\text{ReLU}(z) = \max(0, z)$$
$$\text{ReLU}'(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \leq 0 \end{cases}$$

- **Range**: [0, infinity)
- **Pros**: No vanishing gradient for positive values, computationally efficient
- **Cons**: "Dying ReLU" problem (neurons can become inactive)

#### Softmax (Output layer for multi-class)
$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

- **Range**: (0, 1), sum to 1
- **Use**: Multi-class classification output

---

### 1.5 Loss Functions

#### Binary Cross-Entropy (for binary classification)
$$\mathcal{L} = -\frac{1}{m} \sum_{i=1}^{m} [y^{(i)} \log(\hat{y}^{(i)}) + (1-y^{(i)}) \log(1-\hat{y}^{(i)})]$$

#### Categorical Cross-Entropy (for multi-class classification)
$$\mathcal{L} = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} y_k^{(i)} \log(\hat{y}_k^{(i)})$$

Where $K$ is the number of classes.

---

### 1.6 Backpropagation

Backpropagation computes gradients of the loss with respect to all parameters using the chain rule.

**Output Layer (with softmax and cross-entropy):**
$$dZ^{[L]} = A^{[L]} - Y$$

**For each hidden layer $l = L-1, L-2, ..., 1$:**
$$dZ^{[l]} = (W^{[l+1]})^T \cdot dZ^{[l+1]} \odot \phi'^{[l]}(Z^{[l]})$$

**Parameter gradients:**
$$dW^{[l]} = \frac{1}{m} dZ^{[l]} \cdot (A^{[l-1]})^T$$
$$db^{[l]} = \frac{1}{m} \sum_{i=1}^{m} dZ^{[l]}$$

---

### 1.7 Gradient Descent Variants

#### Batch Gradient Descent
$$W^{[l]} := W^{[l]} - \alpha \cdot dW^{[l]}$$

Uses entire dataset for each update. Stable but slow.

#### Stochastic Gradient Descent (SGD)
Updates after each sample. Fast but noisy.

#### Mini-Batch Gradient Descent
Updates after a batch of samples. Best of both worlds.

#### SGD with Momentum
$$v_t = \beta v_{t-1} + (1-\beta) dW$$
$$W := W - \alpha \cdot v_t$$

Accelerates convergence by accumulating gradient direction.

#### Adam (Adaptive Moment Estimation)
Combines momentum with adaptive learning rates:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) dW$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) dW^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$
$$W := W - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

---

### 1.8 Weight Initialization

Proper initialization prevents vanishing/exploding gradients.

#### Xavier/Glorot Initialization (for tanh/sigmoid)
$$W^{[l]} \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n^{[l-1]} + n^{[l]}}}\right)$$

Or uniform:
$$W^{[l]} \sim \mathcal{U}\left(-\sqrt{\frac{6}{n^{[l-1]} + n^{[l]}}}, \sqrt{\frac{6}{n^{[l-1]} + n^{[l]}}}\right)$$

#### He Initialization (for ReLU)
$$W^{[l]} \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n^{[l-1]}}}\right)$$

---

### Time Complexity

For a network with $L$ layers and $n$ neurons per layer:
- **Forward Pass**: $O(m \cdot n^2 \cdot L)$ per batch
- **Backward Pass**: $O(m \cdot n^2 \cdot L)$ per batch
- **Total Training**: $O(epochs \cdot m \cdot n^2 \cdot L)$

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class MLPClassifier:
    """
    Multi-Layer Perceptron Classifier implemented from scratch using NumPy.
    
    Parameters:
    -----------
    hidden_layers : tuple, default=(64, 32)
        Number of neurons in each hidden layer
    activation : str, default='relu'
        Activation function for hidden layers ('relu', 'tanh', 'sigmoid')
    learning_rate : float, default=0.01
        Learning rate for gradient descent
    max_iter : int, default=200
        Maximum number of iterations (epochs)
    batch_size : int, default=32
        Size of mini-batches for training
    optimizer : str, default='adam'
        Optimization algorithm ('sgd', 'momentum', 'adam')
    momentum : float, default=0.9
        Momentum parameter for SGD with momentum
    beta1 : float, default=0.9
        Adam optimizer beta1 parameter
    beta2 : float, default=0.999
        Adam optimizer beta2 parameter
    epsilon : float, default=1e-8
        Small constant for numerical stability
    l2_lambda : float, default=0.0001
        L2 regularization strength
    early_stopping : bool, default=False
        Whether to use early stopping
    patience : int, default=10
        Number of epochs with no improvement before stopping
    validation_fraction : float, default=0.1
        Fraction of training data for validation
    random_state : int, default=42
        Random seed for reproducibility
    verbose : bool, default=False
        Print progress during training
    """
    
    def __init__(self, hidden_layers=(64, 32), activation='relu', 
                 learning_rate=0.01, max_iter=200, batch_size=32,
                 optimizer='adam', momentum=0.9, beta1=0.9, beta2=0.999,
                 epsilon=1e-8, l2_lambda=0.0001, early_stopping=False,
                 patience=10, validation_fraction=0.1, random_state=42, 
                 verbose=False):
        
        self.hidden_layers = hidden_layers
        self.activation = activation
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.batch_size = batch_size
        self.optimizer = optimizer
        self.momentum = momentum
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.l2_lambda = l2_lambda
        self.early_stopping = early_stopping
        self.patience = patience
        self.validation_fraction = validation_fraction
        self.random_state = random_state
        self.verbose = verbose
        
        # To be initialized during fit
        self.weights = None
        self.biases = None
        self.n_classes = None
        self.n_features = None
        self.layer_sizes = None
        
        # Training history
        self.loss_history = []
        self.train_acc_history = []
        self.val_loss_history = []
        self.val_acc_history = []
        
        # Adam/Momentum optimizer state
        self._m_weights = None
        self._v_weights = None
        self._m_biases = None
        self._v_biases = None
        self._t = 0
    
    # ===========================================
    # ACTIVATION FUNCTIONS
    # ===========================================
    
    def _relu(self, z):
        """ReLU activation function."""
        return np.maximum(0, z)
    
    def _relu_derivative(self, z):
        """Derivative of ReLU."""
        return (z > 0).astype(float)
    
    def _sigmoid(self, z):
        """Sigmoid activation function with numerical stability."""
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def _sigmoid_derivative(self, z):
        """Derivative of sigmoid."""
        s = self._sigmoid(z)
        return s * (1 - s)
    
    def _tanh(self, z):
        """Tanh activation function."""
        return np.tanh(z)
    
    def _tanh_derivative(self, z):
        """Derivative of tanh."""
        return 1 - np.tanh(z) ** 2
    
    def _softmax(self, z):
        """Softmax activation for output layer with numerical stability."""
        # Subtract max for numerical stability
        z_shifted = z - np.max(z, axis=0, keepdims=True)
        exp_z = np.exp(z_shifted)
        return exp_z / np.sum(exp_z, axis=0, keepdims=True)
    
    def _get_activation(self, name):
        """Get activation function and its derivative."""
        activations = {
            'relu': (self._relu, self._relu_derivative),
            'sigmoid': (self._sigmoid, self._sigmoid_derivative),
            'tanh': (self._tanh, self._tanh_derivative)
        }
        return activations[name]
    
    # ===========================================
    # WEIGHT INITIALIZATION
    # ===========================================
    
    def _initialize_weights(self):
        """
        Initialize weights using Xavier/He initialization.
        - Xavier for tanh/sigmoid
        - He for ReLU
        """
        np.random.seed(self.random_state)
        
        self.weights = []
        self.biases = []
        
        for i in range(len(self.layer_sizes) - 1):
            n_in = self.layer_sizes[i]
            n_out = self.layer_sizes[i + 1]
            
            # Choose initialization based on activation
            if self.activation == 'relu':
                # He initialization
                std = np.sqrt(2.0 / n_in)
            else:
                # Xavier initialization
                std = np.sqrt(2.0 / (n_in + n_out))
            
            W = np.random.randn(n_out, n_in) * std
            b = np.zeros((n_out, 1))
            
            self.weights.append(W)
            self.biases.append(b)
        
        # Initialize optimizer state
        self._initialize_optimizer_state()
    
    def _initialize_optimizer_state(self):
        """Initialize optimizer state variables."""
        self._m_weights = [np.zeros_like(W) for W in self.weights]
        self._v_weights = [np.zeros_like(W) for W in self.weights]
        self._m_biases = [np.zeros_like(b) for b in self.biases]
        self._v_biases = [np.zeros_like(b) for b in self.biases]
        self._t = 0
    
    # ===========================================
    # FORWARD PROPAGATION
    # ===========================================
    
    def _forward(self, X):
        """
        Perform forward propagation.
        
        Parameters:
        -----------
        X : ndarray, shape (n_features, n_samples)
            Input data
        
        Returns:
        --------
        A_list : list
            Activations for each layer (including input)
        Z_list : list
            Pre-activation values for each layer
        """
        activation_fn, _ = self._get_activation(self.activation)
        
        A = X
        A_list = [A]  # Store activations (A[0] = X)
        Z_list = []    # Store pre-activations
        
        # Hidden layers
        for i in range(len(self.weights) - 1):
            Z = self.weights[i] @ A + self.biases[i]
            A = activation_fn(Z)
            Z_list.append(Z)
            A_list.append(A)
        
        # Output layer (softmax)
        Z = self.weights[-1] @ A + self.biases[-1]
        A = self._softmax(Z)
        Z_list.append(Z)
        A_list.append(A)
        
        return A_list, Z_list
    
    # ===========================================
    # LOSS COMPUTATION
    # ===========================================
    
    def _compute_loss(self, Y_pred, Y_true):
        """
        Compute cross-entropy loss with L2 regularization.
        
        Parameters:
        -----------
        Y_pred : ndarray, shape (n_classes, n_samples)
            Predicted probabilities
        Y_true : ndarray, shape (n_classes, n_samples)
            One-hot encoded true labels
        
        Returns:
        --------
        loss : float
            Cross-entropy loss with regularization
        """
        m = Y_true.shape[1]
        
        # Clip predictions to prevent log(0)
        Y_pred = np.clip(Y_pred, 1e-15, 1 - 1e-15)
        
        # Cross-entropy loss
        cross_entropy = -np.sum(Y_true * np.log(Y_pred)) / m
        
        # L2 regularization
        l2_reg = 0
        for W in self.weights:
            l2_reg += np.sum(W ** 2)
        l2_reg = (self.l2_lambda / (2 * m)) * l2_reg
        
        return cross_entropy + l2_reg
    
    # ===========================================
    # BACKWARD PROPAGATION
    # ===========================================
    
    def _backward(self, A_list, Z_list, Y_true):
        """
        Perform backward propagation.
        
        Parameters:
        -----------
        A_list : list
            Activations from forward pass
        Z_list : list
            Pre-activations from forward pass
        Y_true : ndarray
            One-hot encoded true labels
        
        Returns:
        --------
        dW_list : list
            Weight gradients for each layer
        db_list : list
            Bias gradients for each layer
        """
        m = Y_true.shape[1]
        _, activation_derivative = self._get_activation(self.activation)
        
        dW_list = []
        db_list = []
        
        # Output layer gradient (softmax + cross-entropy)
        dZ = A_list[-1] - Y_true  # Shape: (n_classes, m)
        
        # Compute gradients from output to input
        for i in range(len(self.weights) - 1, -1, -1):
            # Weight gradient with L2 regularization
            dW = (1/m) * (dZ @ A_list[i].T) + (self.l2_lambda / m) * self.weights[i]
            
            # Bias gradient
            db = (1/m) * np.sum(dZ, axis=1, keepdims=True)
            
            dW_list.insert(0, dW)
            db_list.insert(0, db)
            
            # Propagate gradient to previous layer (if not at input)
            if i > 0:
                dA = self.weights[i].T @ dZ
                dZ = dA * activation_derivative(Z_list[i-1])
        
        return dW_list, db_list
    
    # ===========================================
    # PARAMETER UPDATE (OPTIMIZERS)
    # ===========================================
    
    def _update_parameters(self, dW_list, db_list):
        """
        Update parameters using the specified optimizer.
        """
        if self.optimizer == 'sgd':
            self._sgd_update(dW_list, db_list)
        elif self.optimizer == 'momentum':
            self._momentum_update(dW_list, db_list)
        elif self.optimizer == 'adam':
            self._adam_update(dW_list, db_list)
    
    def _sgd_update(self, dW_list, db_list):
        """Standard SGD update."""
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * dW_list[i]
            self.biases[i] -= self.learning_rate * db_list[i]
    
    def _momentum_update(self, dW_list, db_list):
        """SGD with momentum update."""
        for i in range(len(self.weights)):
            # Update velocity
            self._m_weights[i] = self.momentum * self._m_weights[i] + (1 - self.momentum) * dW_list[i]
            self._m_biases[i] = self.momentum * self._m_biases[i] + (1 - self.momentum) * db_list[i]
            
            # Update parameters
            self.weights[i] -= self.learning_rate * self._m_weights[i]
            self.biases[i] -= self.learning_rate * self._m_biases[i]
    
    def _adam_update(self, dW_list, db_list):
        """Adam optimizer update."""
        self._t += 1
        
        for i in range(len(self.weights)):
            # Update biased first moment estimate
            self._m_weights[i] = self.beta1 * self._m_weights[i] + (1 - self.beta1) * dW_list[i]
            self._m_biases[i] = self.beta1 * self._m_biases[i] + (1 - self.beta1) * db_list[i]
            
            # Update biased second raw moment estimate
            self._v_weights[i] = self.beta2 * self._v_weights[i] + (1 - self.beta2) * (dW_list[i] ** 2)
            self._v_biases[i] = self.beta2 * self._v_biases[i] + (1 - self.beta2) * (db_list[i] ** 2)
            
            # Bias correction
            m_w_hat = self._m_weights[i] / (1 - self.beta1 ** self._t)
            m_b_hat = self._m_biases[i] / (1 - self.beta1 ** self._t)
            v_w_hat = self._v_weights[i] / (1 - self.beta2 ** self._t)
            v_b_hat = self._v_biases[i] / (1 - self.beta2 ** self._t)
            
            # Update parameters
            self.weights[i] -= self.learning_rate * m_w_hat / (np.sqrt(v_w_hat) + self.epsilon)
            self.biases[i] -= self.learning_rate * m_b_hat / (np.sqrt(v_b_hat) + self.epsilon)
    
    # ===========================================
    # TRAINING
    # ===========================================
    
    def _create_mini_batches(self, X, Y, batch_size):
        """Create mini-batches for training."""
        m = X.shape[1]
        mini_batches = []
        
        # Shuffle data
        permutation = np.random.permutation(m)
        X_shuffled = X[:, permutation]
        Y_shuffled = Y[:, permutation]
        
        # Create mini-batches
        n_complete_batches = m // batch_size
        
        for k in range(n_complete_batches):
            X_batch = X_shuffled[:, k * batch_size:(k + 1) * batch_size]
            Y_batch = Y_shuffled[:, k * batch_size:(k + 1) * batch_size]
            mini_batches.append((X_batch, Y_batch))
        
        # Handle remaining samples
        if m % batch_size != 0:
            X_batch = X_shuffled[:, n_complete_batches * batch_size:]
            Y_batch = Y_shuffled[:, n_complete_batches * batch_size:]
            mini_batches.append((X_batch, Y_batch))
        
        return mini_batches
    
    def fit(self, X, y):
        """
        Train the neural network.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target labels
        
        Returns:
        --------
        self : MLPClassifier
            Fitted classifier
        """
        # Convert to numpy and transpose to (n_features, n_samples)
        X = np.array(X).T
        y = np.array(y)
        
        self.n_features = X.shape[0]
        self.n_classes = len(np.unique(y))
        
        # One-hot encode labels
        Y = np.zeros((self.n_classes, X.shape[1]))
        Y[y, np.arange(X.shape[1])] = 1
        
        # Split for validation if early stopping
        if self.early_stopping:
            m = X.shape[1]
            val_size = int(m * self.validation_fraction)
            indices = np.random.permutation(m)
            
            X_val = X[:, indices[:val_size]]
            Y_val = Y[:, indices[:val_size]]
            X = X[:, indices[val_size:]]
            Y = Y[:, indices[val_size:]]
        
        # Define layer sizes
        self.layer_sizes = [self.n_features] + list(self.hidden_layers) + [self.n_classes]
        
        # Initialize weights
        self._initialize_weights()
        
        # Training loop
        best_val_loss = np.inf
        patience_counter = 0
        best_weights = None
        best_biases = None
        
        for epoch in range(self.max_iter):
            # Create mini-batches
            mini_batches = self._create_mini_batches(X, Y, self.batch_size)
            
            epoch_loss = 0
            
            for X_batch, Y_batch in mini_batches:
                # Forward propagation
                A_list, Z_list = self._forward(X_batch)
                
                # Compute loss
                batch_loss = self._compute_loss(A_list[-1], Y_batch)
                epoch_loss += batch_loss * X_batch.shape[1]
                
                # Backward propagation
                dW_list, db_list = self._backward(A_list, Z_list, Y_batch)
                
                # Update parameters
                self._update_parameters(dW_list, db_list)
            
            # Average loss for epoch
            epoch_loss /= X.shape[1]
            self.loss_history.append(epoch_loss)
            
            # Compute training accuracy
            A_train, _ = self._forward(X)
            train_acc = np.mean(np.argmax(A_train[-1], axis=0) == np.argmax(Y, axis=0))
            self.train_acc_history.append(train_acc)
            
            # Validation metrics
            if self.early_stopping:
                A_val, _ = self._forward(X_val)
                val_loss = self._compute_loss(A_val[-1], Y_val)
                val_acc = np.mean(np.argmax(A_val[-1], axis=0) == np.argmax(Y_val, axis=0))
                self.val_loss_history.append(val_loss)
                self.val_acc_history.append(val_acc)
                
                # Early stopping check
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_weights = [W.copy() for W in self.weights]
                    best_biases = [b.copy() for b in self.biases]
                else:
                    patience_counter += 1
                    if patience_counter >= self.patience:
                        if self.verbose:
                            print(f"Early stopping at epoch {epoch}")
                        self.weights = best_weights
                        self.biases = best_biases
                        break
            
            # Print progress
            if self.verbose and epoch % 10 == 0:
                msg = f"Epoch {epoch}: Loss = {epoch_loss:.4f}, Train Acc = {train_acc:.4f}"
                if self.early_stopping:
                    msg += f", Val Acc = {val_acc:.4f}"
                print(msg)
        
        return self
    
    # ===========================================
    # PREDICTION
    # ===========================================
    
    def predict_proba(self, X):
        """
        Predict class probabilities.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Input data
        
        Returns:
        --------
        proba : ndarray, shape (n_samples, n_classes)
            Class probabilities
        """
        X = np.array(X).T
        A_list, _ = self._forward(X)
        return A_list[-1].T  # Transpose back to (n_samples, n_classes)
    
    def predict(self, X):
        """
        Predict class labels.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Input data
        
        Returns:
        --------
        predictions : ndarray, shape (n_samples,)
            Predicted class labels
        """
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)
    
    def score(self, X, y):
        """
        Compute accuracy score.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Input data
        y : array-like, shape (n_samples,)
            True labels
        
        Returns:
        --------
        accuracy : float
            Classification accuracy
        """
        predictions = self.predict(X)
        return np.mean(predictions == y)

In [ ]:
# Quick test of our implementation
print("Testing MLPClassifier implementation...")
print("="*50)

# Create a simple test dataset
X_test_data, y_test_data = make_classification(
    n_samples=200, n_features=10, n_classes=3, 
    n_informative=6, n_redundant=2, random_state=42
)

# Standardize
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test_data)

# Split
X_tr, X_te, y_tr, y_te = train_test_split(X_test_scaled, y_test_data, test_size=0.2, random_state=42)

# Train our model
model = MLPClassifier(
    hidden_layers=(32, 16),
    activation='relu',
    learning_rate=0.01,
    max_iter=100,
    verbose=True
)
model.fit(X_tr, y_tr)

print(f"\nTrain Accuracy: {model.score(X_tr, y_tr):.4f}")
print(f"Test Accuracy: {model.score(X_te, y_te):.4f}")

## 3. Training & Optimization <a id='training'></a>

In [ ]:
# Load the digits dataset (smaller than MNIST, good for CPU training)
digits = load_digits()
X, y = digits.data, digits.target

print("Digits Dataset Information:")
print("="*50)
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]} (8x8 pixel images)")
print(f"Number of classes: {len(np.unique(y))} (digits 0-9)")
print(f"Class distribution: {np.bincount(y)}")

# Display some sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for idx, ax in enumerate(axes.flat):
    ax.imshow(digits.images[idx], cmap='gray')
    ax.set_title(f"Label: {y[idx]}")
    ax.axis('off')
plt.suptitle("Sample Digits from Dataset", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare the data
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape}")
print(f"Test set size: {X_test_scaled.shape}")

In [ ]:
# Train models with different configurations
print("Training Neural Networks with Different Configurations")
print("="*60)

configs = {
    'Shallow Network (64)': {
        'hidden_layers': (64,),
        'activation': 'relu'
    },
    'Deep Network (64, 32)': {
        'hidden_layers': (64, 32),
        'activation': 'relu'
    },
    'Tanh Activation': {
        'hidden_layers': (64, 32),
        'activation': 'tanh'
    },
    'Sigmoid Activation': {
        'hidden_layers': (64, 32),
        'activation': 'sigmoid'
    }
}

trained_models = {}

for name, config in configs.items():
    print(f"\nTraining: {name}")
    print("-" * 40)
    
    model = MLPClassifier(
        hidden_layers=config['hidden_layers'],
        activation=config['activation'],
        learning_rate=0.01,
        max_iter=150,
        batch_size=32,
        optimizer='adam',
        l2_lambda=0.0001,
        verbose=False,
        random_state=42
    )
    
    model.fit(X_train_scaled, y_train)
    
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test Accuracy:  {test_acc:.4f}")
    print(f"  Final Loss:     {model.loss_history[-1]:.4f}")
    
    trained_models[name] = model

In [ ]:
# Compare different optimizers
print("\nComparing Different Optimizers")
print("="*60)

optimizer_configs = {
    'SGD': {'optimizer': 'sgd', 'learning_rate': 0.1},
    'SGD + Momentum': {'optimizer': 'momentum', 'learning_rate': 0.1},
    'Adam': {'optimizer': 'adam', 'learning_rate': 0.01}
}

optimizer_models = {}

for name, config in optimizer_configs.items():
    print(f"\nTraining with {name}...")
    
    model = MLPClassifier(
        hidden_layers=(64, 32),
        activation='relu',
        learning_rate=config['learning_rate'],
        max_iter=150,
        batch_size=32,
        optimizer=config['optimizer'],
        l2_lambda=0.0001,
        verbose=False,
        random_state=42
    )
    
    model.fit(X_train_scaled, y_train)
    
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test Accuracy:  {test_acc:.4f}")
    
    optimizer_models[name] = model

In [ ]:
# Train model with early stopping
print("\nTraining with Early Stopping")
print("="*60)

model_early_stop = MLPClassifier(
    hidden_layers=(64, 32),
    activation='relu',
    learning_rate=0.01,
    max_iter=300,
    batch_size=32,
    optimizer='adam',
    l2_lambda=0.0001,
    early_stopping=True,
    patience=15,
    validation_fraction=0.15,
    verbose=True,
    random_state=42
)

model_early_stop.fit(X_train_scaled, y_train)

print(f"\nFinal Training Accuracy: {model_early_stop.score(X_train_scaled, y_train):.4f}")
print(f"Final Test Accuracy: {model_early_stop.score(X_test_scaled, y_test):.4f}")
print(f"Training stopped at epoch: {len(model_early_stop.loss_history)}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
# Plot loss curves for different architectures
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_models.items()):
    ax = axes[idx]
    ax.plot(model.loss_history, linewidth=2)
    ax.set_title(f"{name}\nFinal Loss: {model.loss_history[-1]:.4f}")
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)

plt.suptitle('Loss Curves for Different Architectures', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot accuracy curves for different architectures
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_models.items()):
    ax = axes[idx]
    ax.plot(model.train_acc_history, linewidth=2, label='Training')
    ax.set_title(f"{name}\nFinal Acc: {model.train_acc_history[-1]:.4f}")
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1.05])

plt.suptitle('Training Accuracy Curves for Different Architectures', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare optimizer convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss comparison
for name, model in optimizer_models.items():
    axes[0].plot(model.loss_history, linewidth=2, label=name)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Convergence by Optimizer')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
for name, model in optimizer_models.items():
    axes[1].plot(model.train_acc_history, linewidth=2, label=name)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy by Optimizer')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1.05])

plt.suptitle('Optimizer Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot early stopping curves
if model_early_stop.early_stopping:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    axes[0].plot(model_early_stop.loss_history, linewidth=2, label='Training Loss')
    axes[0].plot(model_early_stop.val_loss_history, linewidth=2, label='Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training vs Validation Loss (Early Stopping)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[1].plot(model_early_stop.train_acc_history, linewidth=2, label='Training Accuracy')
    axes[1].plot(model_early_stop.val_acc_history, linewidth=2, label='Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training vs Validation Accuracy (Early Stopping)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1.05])
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Select the best model for detailed evaluation
best_model = trained_models['Deep Network (64, 32)']

# Make predictions
y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Neural Network on Digits Dataset')
plt.tight_layout()
plt.show()

# Classification Report
print("\nClassification Report:")
print("="*60)
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

In [ ]:
# Analyze misclassified samples
misclassified_idx = np.where(y_pred != y_test)[0]

print(f"Number of misclassified samples: {len(misclassified_idx)} out of {len(y_test)}")
print(f"Misclassification rate: {len(misclassified_idx)/len(y_test)*100:.2f}%")

# Display some misclassified examples
if len(misclassified_idx) > 0:
    n_display = min(10, len(misclassified_idx))
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    
    for idx, ax in enumerate(axes.flat[:n_display]):
        sample_idx = misclassified_idx[idx]
        # Get the original image (reshape from 64 features to 8x8)
        img = X_test[sample_idx].reshape(8, 8)
        ax.imshow(img, cmap='gray')
        ax.set_title(f"True: {y_test[sample_idx]}\nPred: {y_pred[sample_idx]}")
        ax.axis('off')
    
    # Hide empty subplots if any
    for idx in range(n_display, 10):
        axes.flat[idx].axis('off')
    
    plt.suptitle('Misclassified Samples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-class accuracy analysis
class_accuracies = []
for digit in range(10):
    mask = y_test == digit
    if np.sum(mask) > 0:
        acc = np.mean(y_pred[mask] == y_test[mask])
        class_accuracies.append(acc)
    else:
        class_accuracies.append(0)

plt.figure(figsize=(10, 6))
bars = plt.bar(range(10), class_accuracies, color=sns.color_palette('husl', 10))
plt.axhline(y=np.mean(class_accuracies), color='red', linestyle='--', 
            label=f'Mean: {np.mean(class_accuracies):.3f}')
plt.xlabel('Digit Class')
plt.ylabel('Accuracy')
plt.title('Per-Class Accuracy')
plt.xticks(range(10))
plt.ylim([0, 1.05])
plt.legend()

# Add accuracy labels on bars
for bar, acc in zip(bars, class_accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{acc:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
# Visualize weight matrices
best_model = trained_models['Deep Network (64, 32)']

fig, axes = plt.subplots(1, len(best_model.weights), figsize=(15, 4))

for idx, (W, ax) in enumerate(zip(best_model.weights, axes)):
    im = ax.imshow(W, aspect='auto', cmap='RdBu_r')
    ax.set_title(f'Layer {idx+1} Weights\nShape: {W.shape}')
    ax.set_xlabel('Input Neurons')
    ax.set_ylabel('Output Neurons')
    plt.colorbar(im, ax=ax)

plt.suptitle('Weight Matrices Visualization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize first layer weights as 8x8 images (receptive fields)
# These show what patterns each first-layer neuron is looking for
first_layer_weights = best_model.weights[0]  # Shape: (64, 64) for 64 neurons and 64 input features

# Select some neurons to visualize
n_neurons_to_show = 16
fig, axes = plt.subplots(4, 4, figsize=(10, 10))

for idx, ax in enumerate(axes.flat):
    if idx < n_neurons_to_show:
        # Reshape the weights for this neuron to 8x8
        neuron_weights = first_layer_weights[idx].reshape(8, 8)
        im = ax.imshow(neuron_weights, cmap='RdBu_r')
        ax.set_title(f'Neuron {idx}')
        ax.axis('off')
    else:
        ax.axis('off')

plt.suptitle('First Layer Weights as 8x8 Receptive Fields\n(What Each Neuron "Looks For")', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Weight distribution histograms
fig, axes = plt.subplots(1, len(best_model.weights), figsize=(15, 4))

for idx, (W, ax) in enumerate(zip(best_model.weights, axes)):
    ax.hist(W.flatten(), bins=50, density=True, alpha=0.7, color=sns.color_palette('husl')[idx])
    ax.set_title(f'Layer {idx+1} Weight Distribution\nMean: {W.mean():.4f}, Std: {W.std():.4f}')
    ax.set_xlabel('Weight Value')
    ax.set_ylabel('Density')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Weight Distributions by Layer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Create a 2D dataset for decision boundary visualization
# Use PCA to reduce digits to 2D for visualization
from sklearn.decomposition import PCA

# Reduce to 2D
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_train_scaled)
X_test_2d = pca.transform(X_test_scaled)

print(f"Explained variance ratio: {pca.explained_variance_ratio_.sum():.2%}")

# Train a smaller network on 2D data
model_2d = MLPClassifier(
    hidden_layers=(32, 16),
    activation='relu',
    learning_rate=0.01,
    max_iter=200,
    batch_size=32,
    optimizer='adam',
    verbose=False,
    random_state=42
)
model_2d.fit(X_2d, y_train)

print(f"2D Model Test Accuracy: {model_2d.score(X_test_2d, y_test):.4f}")

In [ ]:
# Plot decision boundaries in 2D
def plot_decision_boundaries(model, X, y, title, ax=None):
    """Plot decision boundaries for 2D data with multiple classes."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 10))
    
    # Create mesh grid
    h = 0.1  # Step size
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='tab10')
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.3)
    
    # Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='tab10', 
                         edgecolors='black', s=40, alpha=0.8)
    
    ax.set_xlabel('PCA Component 1')
    ax.set_ylabel('PCA Component 2')
    ax.set_title(title)
    
    return ax

# Plot decision boundaries
fig, ax = plt.subplots(figsize=(12, 10))
plot_decision_boundaries(model_2d, X_2d, y_train, 
                         'Neural Network Decision Boundaries (2D PCA Projection)', ax=ax)

# Add colorbar/legend
cbar = plt.colorbar(ax.collections[1], ax=ax, ticks=range(10))
cbar.set_label('Digit Class')

plt.tight_layout()
plt.show()

In [ ]:
# Activation visualization - showing what neurons activate for different inputs
def get_activations(model, X):
    """Get activations for all layers."""
    X = np.array(X).T
    A_list, _ = model._forward(X)
    return [A.T for A in A_list]  # Transpose back to (n_samples, n_neurons)

# Get activations for test set
activations = get_activations(best_model, X_test_scaled)

# Plot mean activation per class for first hidden layer
first_hidden_activations = activations[1]  # Shape: (n_samples, 64)

# Compute mean activation per class
mean_activations_per_class = []
for digit in range(10):
    mask = y_test == digit
    mean_act = first_hidden_activations[mask].mean(axis=0)
    mean_activations_per_class.append(mean_act)

mean_activations_per_class = np.array(mean_activations_per_class)

plt.figure(figsize=(14, 6))
sns.heatmap(mean_activations_per_class, cmap='YlOrRd', 
            yticklabels=[f'Digit {i}' for i in range(10)])
plt.xlabel('Hidden Neuron Index')
plt.ylabel('Digit Class')
plt.title('Mean First Hidden Layer Activations by Digit Class')
plt.tight_layout()
plt.show()

In [ ]:
# Prediction confidence analysis
proba = best_model.predict_proba(X_test_scaled)
max_proba = proba.max(axis=1)  # Confidence of prediction

# Split by correct/incorrect predictions
correct_mask = y_pred == y_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution
axes[0].hist(max_proba[correct_mask], bins=30, alpha=0.7, label='Correct', color='green')
axes[0].hist(max_proba[~correct_mask], bins=30, alpha=0.7, label='Incorrect', color='red')
axes[0].set_xlabel('Prediction Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Confidence Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Confidence vs accuracy
confidence_bins = np.linspace(0, 1, 11)
bin_accuracies = []
bin_counts = []

for i in range(len(confidence_bins) - 1):
    mask = (max_proba >= confidence_bins[i]) & (max_proba < confidence_bins[i+1])
    if mask.sum() > 0:
        bin_accuracies.append(correct_mask[mask].mean())
        bin_counts.append(mask.sum())
    else:
        bin_accuracies.append(0)
        bin_counts.append(0)

bin_centers = (confidence_bins[:-1] + confidence_bins[1:]) / 2

axes[1].bar(bin_centers, bin_accuracies, width=0.08, alpha=0.7)
axes[1].plot([0, 1], [0, 1], 'r--', label='Perfect Calibration')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Calibration Plot (Confidence vs Accuracy)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.suptitle('Model Confidence Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use Neural Networks (MLP)

#### Ideal Use Cases:

1. **Complex Non-Linear Patterns**
   - When decision boundaries are highly non-linear
   - When features have complex interactions
   - Image recognition, speech processing, natural language processing

2. **Large Datasets**
   - Neural networks excel with large amounts of data
   - More data = better generalization
   - Rule of thumb: At least 10x samples per weight parameter

3. **High-Dimensional Input**
   - Images (thousands of pixels)
   - Text (large vocabularies)
   - Sensor data with many channels

4. **Feature Learning**
   - When manual feature engineering is difficult
   - The network learns useful representations automatically

5. **Multi-Class Classification**
   - Naturally handles many classes via softmax
   - No need for one-vs-rest strategies

---

### When NOT to Use Neural Networks

#### Avoid When:

1. **Small Datasets**
   - High risk of overfitting
   - Better alternatives: Logistic Regression, SVM, Random Forest
   - Rule of thumb: < 1000 samples may be too small

2. **Interpretability is Critical**
   - Neural networks are "black boxes"
   - Use: Decision Trees, Logistic Regression for explainability
   - Domains: Healthcare, finance, legal (where explanations required)

3. **Limited Computational Resources**
   - Training can be slow without GPU
   - Consider simpler models for resource-constrained environments

4. **Linear or Simple Relationships**
   - Overkill for linearly separable data
   - Logistic Regression will be faster and equally accurate

5. **Real-Time Training Requirements**
   - Online learning scenarios with instant model updates
   - Consider: Naive Bayes, Online Gradient Descent methods

---

### Hyperparameter Tuning Guidelines

| Parameter | Typical Range | Tuning Strategy |
|-----------|---------------|------------------|
| Hidden Layers | 1-5 layers | Start small, increase if underfitting |
| Neurons per Layer | 32-512 | Powers of 2, wider = more capacity |
| Learning Rate | 0.0001-0.1 | Start with 0.01, use learning rate finder |
| Batch Size | 16-256 | Larger = stable gradients, smaller = faster updates |
| Epochs | 50-500 | Use early stopping to find optimal |
| L2 Regularization | 0.0001-0.01 | Higher if overfitting |
| Dropout | 0.1-0.5 | Use between layers, higher for larger networks |

---

### Architecture Design Rules of Thumb

1. **Start Simple**: Begin with one hidden layer, add complexity if needed
2. **Funnel Shape**: Decreasing layer sizes (e.g., 128 -> 64 -> 32)
3. **Same Size**: Sometimes equal-sized layers work well
4. **Output Layer**: Match the number of classes (softmax for multi-class)

---

### Avoiding Overfitting

1. **Regularization Techniques**
   - L2 regularization (weight decay)
   - Dropout (randomly zero out neurons during training)
   - Early stopping (stop when validation loss increases)

2. **Data Augmentation**
   - Create synthetic training examples
   - Rotations, flips, noise for images
   - Synonyms, paraphrasing for text

3. **Reduce Model Complexity**
   - Fewer layers or neurons
   - Smaller batch sizes (more noise = regularization)

4. **Ensemble Methods**
   - Train multiple networks
   - Average their predictions

---

### Avoiding Underfitting

1. **Increase Model Capacity**
   - Add more layers or neurons
   - Use more complex activation functions

2. **Train Longer**
   - More epochs may be needed
   - Monitor loss curve for convergence

3. **Reduce Regularization**
   - Lower L2 lambda
   - Reduce dropout rate

4. **Feature Engineering**
   - Add more informative features
   - Create polynomial features

---

### Practical Tips

1. **Always Normalize/Standardize Input**
   - Neural networks are sensitive to feature scales
   - Use StandardScaler or MinMaxScaler

2. **Initialize Weights Properly**
   - Xavier for tanh/sigmoid
   - He for ReLU

3. **Use Adam Optimizer**
   - Good default choice
   - Adapts learning rate automatically

4. **Monitor Training/Validation Curves**
   - Diverging curves = overfitting
   - Both high = underfitting

5. **Batch Normalization** (advanced)
   - Normalizes layer inputs
   - Allows higher learning rates
   - Acts as regularization

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare with sklearn's MLPClassifier
print("Comparison: Our Implementation vs sklearn MLPClassifier")
print("="*60)

# Our implementation
our_model = MLPClassifier(
    hidden_layers=(64, 32),
    activation='relu',
    learning_rate=0.01,
    max_iter=200,
    batch_size=32,
    optimizer='adam',
    l2_lambda=0.0001,
    random_state=42,
    verbose=False
)

# sklearn implementation
sklearn_model = SklearnMLP(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    learning_rate_init=0.01,
    max_iter=200,
    batch_size=32,
    solver='adam',
    alpha=0.0001,
    random_state=42,
    verbose=False
)

# Train both models
import time

print("\nTraining Our Implementation...")
start_time = time.time()
our_model.fit(X_train_scaled, y_train)
our_time = time.time() - start_time

print("Training sklearn MLPClassifier...")
start_time = time.time()
sklearn_model.fit(X_train_scaled, y_train)
sklearn_time = time.time() - start_time

# Evaluate both
our_train_acc = our_model.score(X_train_scaled, y_train)
our_test_acc = our_model.score(X_test_scaled, y_test)
our_pred = our_model.predict(X_test_scaled)

sklearn_train_acc = sklearn_model.score(X_train_scaled, y_train)
sklearn_test_acc = sklearn_model.score(X_test_scaled, y_test)
sklearn_pred = sklearn_model.predict(X_test_scaled)

# Print comparison
print("\n" + "="*60)
print(f"{'Metric':<25} {'Our Model':<15} {'sklearn':<15}")
print("="*60)
print(f"{'Training Time (s)':<25} {our_time:<15.4f} {sklearn_time:<15.4f}")
print(f"{'Training Accuracy':<25} {our_train_acc:<15.4f} {sklearn_train_acc:<15.4f}")
print(f"{'Test Accuracy':<25} {our_test_acc:<15.4f} {sklearn_test_acc:<15.4f}")
print(f"{'F1 Score (weighted)':<25} {f1_score(y_test, our_pred, average="weighted"):<15.4f} {f1_score(y_test, sklearn_pred, average="weighted"):<15.4f}")
print("="*60)

In [ ]:
# Compare loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Our model loss
axes[0].plot(our_model.loss_history, linewidth=2, label='Our Implementation')
axes[0].plot(sklearn_model.loss_curve_, linewidth=2, label='sklearn', linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Prediction agreement
agreement = (our_pred == sklearn_pred).astype(int)
cm_agreement = confusion_matrix(sklearn_pred, our_pred)

sns.heatmap(cm_agreement, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=range(10), yticklabels=range(10))
axes[1].set_xlabel('Our Predictions')
axes[1].set_ylabel('sklearn Predictions')
axes[1].set_title(f'Prediction Agreement Matrix\n(Agreement: {agreement.mean():.2%})')

plt.suptitle('Our Implementation vs sklearn MLPClassifier', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare probability predictions
our_proba = our_model.predict_proba(X_test_scaled)
sklearn_proba = sklearn_model.predict_proba(X_test_scaled)

# Compute probability differences
proba_diff = np.abs(our_proba - sklearn_proba)
mean_diff_per_class = proba_diff.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot of probabilities for a specific class
class_idx = 0
axes[0].scatter(sklearn_proba[:, class_idx], our_proba[:, class_idx], alpha=0.5, s=20)
axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect Agreement')
axes[0].set_xlabel(f'sklearn P(class={class_idx})')
axes[0].set_ylabel(f'Our P(class={class_idx})')
axes[0].set_title(f'Probability Comparison for Class {class_idx}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mean probability difference per class
axes[1].bar(range(10), mean_diff_per_class, color=sns.color_palette('husl', 10))
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Mean Absolute Probability Difference')
axes[1].set_title('Mean Probability Difference by Class')
axes[1].set_xticks(range(10))
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Probability Comparison (Overall MAE: {proba_diff.mean():.4f})', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Our model
cm_our = confusion_matrix(y_test, our_pred)
sns.heatmap(cm_our, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=range(10), yticklabels=range(10))
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title(f'Our Implementation\n(Accuracy: {our_test_acc:.4f})')

# sklearn model
cm_sklearn = confusion_matrix(y_test, sklearn_pred)
sns.heatmap(cm_sklearn, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=range(10), yticklabels=range(10))
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title(f'sklearn MLPClassifier\n(Accuracy: {sklearn_test_acc:.4f})')

plt.suptitle('Confusion Matrix Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary & Key Takeaways

### What We Learned:

1. **Neural Network Architecture**
   - Perceptrons are the basic building blocks
   - Multiple layers enable learning complex patterns
   - Depth vs. width tradeoffs

2. **Forward & Backward Propagation**
   - Forward pass computes predictions layer by layer
   - Backpropagation efficiently computes gradients using chain rule
   - Proper implementation requires careful matrix operations

3. **Activation Functions**
   - ReLU is the modern default for hidden layers
   - Softmax for multi-class output
   - Each has tradeoffs (vanishing gradients, dying neurons)

4. **Optimization**
   - Adam optimizer is a good default choice
   - Learning rate is crucial - not too high, not too low
   - Mini-batch training balances speed and stability

5. **Regularization & Generalization**
   - L2 regularization prevents large weights
   - Early stopping prevents overfitting
   - Monitor validation metrics during training

### Key Insights:

- **Weight initialization matters**: Xavier/He initialization prevents vanishing/exploding gradients
- **Feature scaling is essential**: Always standardize inputs
- **Architecture search is empirical**: Start simple, increase complexity as needed
- **Our implementation achieves comparable performance to sklearn**: Validates our understanding

### Next Steps:

1. **Advanced Architectures**: Convolutional Neural Networks (CNNs), Recurrent Neural Networks (RNNs)
2. **Regularization Techniques**: Dropout, Batch Normalization
3. **Learning Rate Schedulers**: Reduce on plateau, cosine annealing
4. **GPU Acceleration**: Use frameworks like PyTorch or TensorFlow for large-scale training